# 第 15 课｜时间序列与平稳性

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`15_youtube-yro4gb2hNOg/index.html`  
> **所属模块**：P1-M3 · 统计证据与可预测性

## 本节量化问题

为什么直接回归价格水平容易得到看似显著但不可用的关系？

## 准确定义与策略报告位置

弱平稳要求均值、方差和协方差结构不随绝对时间变化；差分或收益化常用于去除随机趋势。

- **核心关系**：弱平稳要求均值、方差和只依赖滞后的协方差稳定；ADF 检验原假设为存在单位根。
- **写入报告**：序列变换、平稳性和伪回归风险
- **P2 衔接**：P2 第 35–36、49 课：时间索引、窗口和回归。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 分别检验价格水平和收益率

ADF 原假设是存在单位根。这里比较对数价格与其一阶差分，p 值不能单独证明经济关系。

In [2]:
from statsmodels.tsa.stattools import adfuller

daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
market = daily.loc[daily["symbol"] == "SIM_MARKET"].set_index("date")["adj_close"]
log_price = np.log(market)
log_return = log_price.diff().dropna()

price_adf = adfuller(log_price, autolag="AIC")
return_adf = adfuller(log_return, autolag="AIC")
display(pd.DataFrame({
    "series": ["log_price", "log_return"],
    "adf_stat": [price_adf[0], return_adf[0]],
    "pvalue": [price_adf[1], return_adf[1]],
    "used_lags": [price_adf[2], return_adf[2]],
}))

,series,adf_stat,pvalue,used_lags
0,log_price,-2.285442,0.176706,1
1,log_return,-26.198365,0.000000,0


## 方法边界与阶段验收

ADF 未拒绝单位根不等于确认随机游走；结构突变会影响检验结论。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
